# 159 — Token Rank Analysis

How do token rankings evolve as the model processes? Track specific tokens
through the vocabulary ranking at each layer, measure rank stability, and
identify competing predictions.

## Why This Matters

Token rank tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_rank_analysis import (
    token_rank_trajectory,
    rank_stability,
    rank_entropy,
    competing_tokens,
    rank_change_attribution,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = token_rank_trajectory(model, tokens)
print(f"Tracked tokens: {result['tracked_tokens']}\n")
for p in result['per_layer']:
    ranks_str = '  '.join(f'tok{t}@{p["ranks"][t]}' for t in result['tracked_tokens'])
    print(f"Layer {p['layer']}: top={p['top_token']}  {ranks_str}")

In [ ]:
result = rank_stability(model, tokens, top_k=10)
print(f"Mean stability: {result['mean_stability']:.3f}\n")
for t in result['transitions']:
    print(f"L{t['from_layer']}->L{t['to_layer']}: overlap={t['overlap']}/10  stability={t['stability']:.3f}")

In [ ]:
result = rank_entropy(model, tokens)
print(f"Entropy reduction: {result['entropy_reduction']:.4f}\n")
for p in result['per_layer']:
    bar = '#' * int(p['normalized_entropy'] * 30)
    print(f"Layer {p['layer']}: H={p['entropy']:.2f} (norm={p['normalized_entropy']:.3f})  "
          f"max_p={p['max_probability']:.4f}  {bar}")

In [ ]:
result = competing_tokens(model, tokens, margin=3.0)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: {p['n_competitors']} competitors  "
          f"top={p['top_token']}  logit={p['top_logit']:.4f}")

In [ ]:
result = rank_change_attribution(model, tokens)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    print(f"Layer {p['layer']}: attn={p['attn_logit_change']:+.4f} mlp={p['mlp_logit_change']:+.4f} "
          f"total={p['total_change']:+.4f} driver={p['main_driver']}")